

Rossmann operates over 3,000 drug stores in 7 European countries. Rossmann is challenging you to predict 6 weeks of daily sales for 1,115 stores located across Germany.

In [1]:
pip install pandas numpy seaborn plotly matplotlib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
file_path = '/content/drive/My Drive/Data Science Projects/Sales_Forecast/train.csv'
train_df = pd.read_csv(file_path)
train_df.head(10)

/tmp/ipykernel_1012/634720609.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(file_path)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1
5,6,5,2015-07-31,5651,589,1,1,0,1
6,7,5,2015-07-31,15344,1414,1,1,0,1
7,8,5,2015-07-31,8492,833,1,1,0,1
8,9,5,2015-07-31,8565,687,1,1,0,1
9,10,5,2015-07-31,7185,681,1,1,0,1


In [ ]:
file_path = '/content/drive/My Drive/Data Science Projects/Sales_Forecast/store.csv'
store_df = pd.read_csv(file_path)
store_df.head(5)

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [ ]:
file_path = '/content/drive/My Drive/Data Science Projects/Sales_Forecast/test.csv'
test_df = pd.read_csv(file_path)
test_df.head(5)

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


Data exploration to find insights

In [ ]:
print(train_df.shape)
print(test_df.shape)
print(store_df.shape)

(1017209, 9)
(41088, 8)
(1115, 10)


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  object
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 69.8+ MB


Join train with store

In [ ]:
Ml_train_df = pd.merge(train_df, store_df, on='Store', how='left')
print(len(Ml_train_df))
Ml_train_df.head(5)

1017209


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


Data cleaning

Handling duplicates: There is no unique identifier in train dataset so there is no column for investigating duplicates

Handling null values for categorical values

In [ ]:
Ml_train_df['StateHoliday'] = Ml_train_df['StateHoliday'].astype(str)

categorical_features = ['StoreType', 'Assortment', 'StateHoliday', 'PromoInterval']
for feature in categorical_features:
    Ml_train_df[feature] = Ml_train_df[feature].fillna('Unknown')

print(Ml_train_df[categorical_features].isnull().sum())

Null values after cleaning categorical features in Ml_train_df:
StoreType        0
Assortment       0
StateHoliday     0
PromoInterval    0
dtype: int64



Handling missing values in numerical data

In [ ]:
# Impute missing numerical values
# For CompetitionDistance, Promo2SinceWeek, Promo2SinceYear, CompetitionOpenSinceMonth, CompetitionOpenSinceYear
numerical_features = ['CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear',]
for feature in numerical_features:
    Ml_train_df[feature] = Ml_train_df[feature].fillna(Ml_train_df[feature].median())

print(Ml_train_df[numerical_features].isnull().sum())

CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2SinceWeek              0
Promo2SinceYear              0
dtype: int64


ml_test_data

In [ ]:
Ml_test_data = pd.merge(test_df, store_df, on='Store', how='left')
print(len(Ml_test_data))
Ml_test_data.head(5)

41088


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,1,4,2015-09-17,1.0,1,0,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,3,4,2015-09-17,1.0,1,0,0,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
2,3,7,4,2015-09-17,1.0,1,0,0,a,c,24000.0,4.0,2013.0,0,NaN,NaN,NaN
3,4,8,4,2015-09-17,1.0,1,0,0,a,a,7520.0,10.0,2014.0,0,NaN,NaN,NaN
4,5,9,4,2015-09-17,1.0,1,0,0,a,c,2030.0,8.0,2000.0,0,NaN,NaN,NaN


Data visualisation

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Convert 'order_date' to datetime objects
Ml_train_df['Date'] = pd.to_datetime(Ml_train_df['Date'])

# Aggregate sales by month
Ml_train_df['Date_year'] = Ml_train_df['Date'].dt.to_period('Y')
yearly_sales = Ml_train_df.groupby('Date_year')['Sales'].sum().reset_index()
yearly_sales['Date_year'] = yearly_sales['Date_year'].astype(str)

# Create a Plotly figure for monthly sales
fig = go.Figure()
fig.add_trace(go.Scatter(x=yearly_sales['Date_year'], y=yearly_sales['Sales'], mode='lines+markers', name='Yearly Sales'))
fig.update_layout(title='Yearly Sales Over Time', xaxis_title='Year', yaxis_title='Total Sales', template='plotly_dark')
fig.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Convert 'order_date' to datetime objects
Ml_train_df['Date'] = pd.to_datetime(Ml_train_df['Date'])

# Aggregate sales by month
Ml_train_df['Date_Month'] = Ml_train_df['Date'].dt.to_period('M')
Monthly_sales = Ml_train_df.groupby('Date_Month')['Sales'].sum().reset_index()
Monthly_sales['Date_Month'] = Monthly_sales['Date_Month'].astype(str)

# Create a Plotly figure for monthly sales
fig = go.Figure()
fig.add_trace(go.Scatter(x=Monthly_sales['Date_Month'], y=Monthly_sales['Sales'], mode='lines+markers', name='Monthly Sales'))
fig.update_layout(title='Monthly Sales Over Time', xaxis_title='Year', yaxis_title='Total Sales', template='plotly_dark')
fig.show()

exporting test and train data for statistical analysis

In [ ]:
from google.colab import files

# Save the DataFrame to a CSV file
output_filename1 = 'Ml_train_sales_forecast_data.csv'
Ml_train_df.to_csv(output_filename1, index=False)
output_filename2 = 'Ml_test_sales_forecast_data.csv'
Ml_test_data.to_csv(output_filename2, index=False)
# Download the file to the local machine
files.download(output_filename1)
files.download(output_filename2)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>